# Lab 8.3 &mdash; Data Boundaries: Prompt, Trace, Vector Store

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 8 &mdash; Safety &amp; Guardrails**

### What you'll do
- Decide the boundary once &mdash; allow-list or block-list &mdash; and put it in the tool
- Point the same decision at a real LangChain callback handler: the trace is a store
- Check <code>Document</code>s before they are embedded &mdash; the store you cannot un-write
- Give every store a retention number somebody chose

> **How this lab works.** You write real Pydantic, LangChain and LangGraph code. Fill every
> `BLANK`, then run the **Self-check** cell under each section &mdash; those assert on the
> *objects you built*: a contract that refuses, a tool that refuses, a compiled graph with a
> gate in it. Refusal is deterministic, so none of it needs the model. Cells marked
> **Run it for real** put your guardrail in front of the sandbox model; that is the part worth
> watching. The score line is feedback, not a grade.

> **Three data stores, and you planned one of them.** The trace and the index are the
> ones that turn up in a review, and neither has anything to do with the model.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-8-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)


def _blank_underneath(exc: BaseException) -> bool:
    """Is an unfilled blank the real cause of this exception?

    A framework -- LangGraph, a tool runner, a parser -- may catch and re-raise what your
    node raised. If the NameError from an unfilled blank arrives wrapped, [TODO] would
    silently become [FAIL]: 'your answer is wrong' instead of 'you have not written one'.
    """
    seen, cur = 0, exc
    while cur is not None and seen < 10:
        if isinstance(cur, NameError):
            return True
        if "'BLANK' is not defined" in str(cur):
            return True
        cur = cur.__cause__ or cur.__context__
        seen += 1
    return False


def unblanked(fn: Callable, *args, **kwargs) -> Any:
    """Call fn(...). If an unfilled blank is underneath -- even wrapped by a framework --
    re-raise it as a plain NameError, so check() prints [TODO] rather than [FAIL]."""
    try:
        return fn(*args, **kwargs)
    except NameError:
        raise
    except Exception as exc:
        if _blank_underneath(exc):
            raise NameError("an unfilled blank is underneath: " + str(exc)[:80])
        raise


def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default
    except Exception as exc:
        if _blank_underneath(exc):
            print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
            return default
        raise


def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and that reasoning is billed as completion
# tokens. Off is the default here because the live cells in this module make a lot of calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the ledger, with what is really in it
# The same payments, as the upstream system actually returns them. Everything below the
# divider in each record is customer data the agent has no use for. Synthetic throughout.

RAW_LEDGER = {
    "PMT-1003": {
        "ref": "PMT-1003", "amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
        "status": "held", "reason_code": "LIMIT_BREACH",
        # ---- customer data ----
        "beneficiary_name": "A. Sharma",
        "beneficiary_iban": "GB29NWBK60161331926819",
        "originator_account": "0021447788",
        "contact_email": "a.sharma@example.com",
        "contact_phone": "+44 7700 900123",
        "internal_memo": "client called, very unhappy",
    },
    "PMT-1005": {
        "ref": "PMT-1005", "amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
        "status": "held", "reason_code": "SANCTIONS_REVIEW",
        # ---- customer data ----
        "beneficiary_name": "L. Okonkwo",
        "beneficiary_iban": "DE89370400440532013000",
        "originator_account": "0098221133",
        "contact_email": "l.okonkwo@example.com",
        "contact_phone": "+44 7700 900456",
        "internal_memo": "second escalation this month",
    },
}

AGENT_FIELDS = ("ref", "amount", "ccy", "counterparty", "status", "reason_code")

PII_FIELDS = ("beneficiary_name", "beneficiary_iban", "originator_account",
              "contact_email", "contact_phone", "internal_memo")

PII_VALUES = tuple(str(rec[f]) for rec in RAW_LEDGER.values() for f in PII_FIELDS)

def leaks(payload) -> list:
    """Which customer values appear anywhere in this payload, once it is serialised."""
    blob = json.dumps(payload, default=str).lower()
    return sorted({v for v in PII_VALUES if v.lower() in blob})

print(f"{len(RAW_LEDGER)} records, {len(PII_FIELDS)} customer fields each")

## Concept

Everyone thinks about what goes into the prompt. Two other stores fill up quietly:

- the **trace**, which keeps every tool input and output, searchable, for as long as retention says
- the **vector store**, which keeps whatever was ingested, in chunks, and is awkward to un-write

The fix is one decision, made once and pointed at all three: what crosses the boundary. Everything
in this lab is that same decision, wearing three different framework objects.

## Section 1 &mdash; Decide the boundary, then put it in the tool

The tool is where data enters the agent's world, so it is where the boundary belongs. Redacting
later &mdash; in the prompt template, in the trace exporter &mdash; means the data was already
somewhere before you looked at it.

In [ ]:
from langchain_core.tools import tool

def keep(field: str, allow=AGENT_FIELDS, deny=PII_FIELDS) -> bool:
    """Decide whether one field survives the boundary into the agent's world.

    Both lists are right in front of you and both look correct today. Only one of them is
    still correct next year, when somebody adds a column to this table and tells nobody.
    """
    # TODO: allow-list or block-list? Write the one that stays correct.
    return BLANK


def redact(record: dict) -> dict:
    """Apply that decision to a whole record."""
    return {k: v for k, v in record.items() if keep(k)}


@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1003'.

    Only the fields an operations decision turns on are returned.
    """
    record = RAW_LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps(redact(record))


@tool
def raw_lookup(ref: str) -> str:
    """Return the whole ledger record for one payment reference such as 'PMT-1003'.

    This is the version somebody writes first, because it is the version the API returns.
    """
    return json.dumps(RAW_LEDGER.get(ref, {}), default=str)

In [ ]:
# --- Self-check: Section 1   (two tool objects, invoked directly -- no model call)
def _out(t, ref="PMT-1003") -> str:
    return unblanked(t.invoke, {"ref": ref})

check("the raw tool hands over every customer field",
      lambda: len(leaks(_out(raw_lookup))) >= 5,
      "this is what the tool the API documentation suggests actually returns")
check("THE REDACTED TOOL LEAKS NOTHING",
      lambda: leaks(_out(lookup_payment)) == [])
check("and still carries what the decision turns on",
      lambda: json.loads(_out(lookup_payment))["reason_code"] == "LIMIT_BREACH")
check("it carries exactly the agent fields, no more",
      lambda: set(json.loads(_out(lookup_payment))) == set(AGENT_FIELDS))
check("A COLUMN ADDED NEXT YEAR IS DROPPED, with nobody updating a list",
      lambda: keep("passport_no") is False,
      "a block-list lets this through -- it is a list of the leaks you already thought of")
check("the boundary is the same for every record, not tuned per case",
      lambda: leaks(_out(lookup_payment, "PMT-1005")) == [])
check("the tool still describes itself to the model",
      lambda: "PMT-1003" in (lookup_payment.description or ""),
      "a redacted tool is still a tool; the description is how the model knows to call it")

guard(lambda: print("  agent sees:", _out(lookup_payment)))

## Section 2 &mdash; The trace is a data store

Module 7's tracer recorded inputs and outputs. In LangChain that is a `BaseCallbackHandler`, and
it sees the tool's arguments and its return value &mdash; before anything you did to the prompt.

Point the same decision at it. Whatever reaches `self.spans` is persisted, searchable, and
outlives the run.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class RedactingTracer(BaseCallbackHandler):
    """Module 7's tracer, with a boundary on it.

    LangChain calls on_tool_start / on_tool_end for any tool invoked with this handler
    attached. What those methods append is what the trace store keeps.
    """
    def __init__(self, redacting: bool = True):
        self.spans = []
        self.redacting = redacting

    def on_tool_start(self, serialized, input_str, **kwargs):
        self.spans.append({"event": "tool.start", "payload": self.record(input_str)})

    def on_tool_end(self, output, **kwargs):
        self.spans.append({"event": "tool.end", "payload": self.record(output)})

    def record(self, blob):
        """The one place that decides what the trace store keeps."""
        try:
            payload = json.loads(blob)
        except (TypeError, ValueError):
            return str(blob)[:200]
        if not self.redacting or not isinstance(payload, dict):
            return payload
        # TODO: the trace is a data store too. Same decision as Section 1, pointed here.
        return BLANK


def traced(redacting: bool = True):
    """A tracer, and the config that attaches it to any Runnable."""
    t = RedactingTracer(redacting=redacting)
    return t, {"callbacks": [t]}

In [ ]:
# --- Self-check: Section 2   (the handler, driven by hand -- no model, no runnable)
def _spans(redacting: bool) -> list:
    """Feed the tracer exactly what a tool call would, and read back what it kept."""
    t = RedactingTracer(redacting=redacting)
    t.on_tool_start({"name": "raw_lookup"}, json.dumps({"ref": "PMT-1003"}))
    t.on_tool_end(json.dumps(RAW_LEDGER["PMT-1003"], default=str))
    return t.spans

check("the tracer is a real LangChain callback handler",
      lambda: isinstance(RedactingTracer(), BaseCallbackHandler),
      "which is why it can be attached to anything, including tools you did not write")
check("AN UNREDACTED TRACER COPIES THE CUSTOMER RECORD INTO THE TRACE STORE",
      lambda: len(leaks(_spans(redacting=False))) >= 5,
      "an observability improvement, and a copy of the customer database")
check("a redacting tracer keeps the span and drops the data",
      lambda: leaks(_spans(redacting=True)) == [])
check("the trace still says which case it was",
      lambda: any("PMT-1003" in json.dumps(s, default=str) for s in _spans(True)),
      "you can debug from a redacted trace; you cannot un-write an unredacted one")
check("both events are recorded either way",
      lambda: len(_spans(True)) == 2 and len(_spans(False)) == 2)
check("it is the SAME decision, applied to a different store",
      lambda: _spans(True)[-1]["payload"] == redact(RAW_LEDGER["PMT-1003"]))

def _real_callback():
    """Attach it to a real tool call and see LangChain drive it for you."""
    t, cfg = traced(redacting=True)
    raw_lookup.invoke({"ref": "PMT-1005"}, config=cfg)
    if not t.spans:
        # LangChain logs a failing callback and carries on, so an unfilled blank in
        # record() shows up here as silence rather than as an error.
        print("  no spans recorded -- record() above still has an unfilled blank")
        return
    print(f"  {len(t.spans)} span(s) recorded by LangChain")
    for s in t.spans:
        print("   ", s["event"], json.dumps(s["payload"], default=str)[:88])
    print("  leaked into the trace:", leaks(t.spans) or "nothing")
    print("  ...and note the TOOL was the unredacted one. The boundary held anyway.")
guard(_real_callback)

## Section 3 &mdash; The index you cannot un-write

A trace expires. An embedded chunk sits in the index until somebody re-indexes, retrievable by
everyone the retriever serves. Check what went in *before* it goes in &mdash; on a
`Document`, which is the object every LangChain loader and splitter hands you.

In [ ]:
from langchain_core.documents import Document

DOCS_TO_INDEX = [
    Document(page_content="Payments above USD 500,000 require Treasury approval.",
             metadata={"source": "runbook-v4.md"}),
    Document(page_content="A payment held for SANCTIONS_REVIEW is decided by Compliance.",
             metadata={"source": "runbook-v4.md"}),
    # somebody exported a case file into the knowledge base
    Document(page_content=("PMT-1003 beneficiary A. Sharma, IBAN GB29NWBK60161331926819, "
                           "called and was unhappy."),
             metadata={"source": "case-notes.md"}),
]

INDEXABLE_SOURCES = {"runbook-v4.md", "policy-v2.md"}

def safe_to_index(doc: Document) -> bool:
    """Two conditions, and BOTH must hold before anything is embedded.

    The source check is cheap and runs before you read a word. The content check is there
    because somebody will paste a real case into the runbook.
    """
    return (doc.metadata.get("source") in INDEXABLE_SOURCES
            and leaks(doc.page_content) == [])


def index_report() -> dict:
    ok = [d for d in DOCS_TO_INDEX if safe_to_index(d)]
    return {"indexed": len(ok),
            "rejected": [d.metadata["source"] for d in DOCS_TO_INDEX if not safe_to_index(d)]}

In [ ]:
# --- Self-check: Section 3   (Document objects -- no embedding, no store, no model)
check("the two runbook chunks are safe to index",
      lambda: index_report()["indexed"] == 2)
check("THE EXPORTED CASE FILE IS REJECTED",
      lambda: index_report()["rejected"] == ["case-notes.md"])
check("it would be rejected on its SOURCE alone",
      lambda: safe_to_index(Document(page_content="nothing sensitive here",
                                     metadata={"source": "case-notes.md"})) is False,
      "an allow-list of sources is the cheap check, and it runs before you read a word")
check("and on its CONTENT alone, even from an allowed source",
      lambda: safe_to_index(Document(page_content="example: IBAN GB29NWBK60161331926819",
                                     metadata={"source": "runbook-v4.md"})) is False,
      "belt and braces, because somebody will paste a real case into the runbook")
check("both conditions are required, not either",
      lambda: safe_to_index(Document(page_content="clean",
                                     metadata={"source": "runbook-v4.md"})) is True)
check("a Document with no source at all is not indexed",
      lambda: safe_to_index(Document(page_content="clean")) is False,
      "unknown provenance is not a reason to proceed")

def _index():
    r = index_report()
    print(f"  indexed {r['indexed']} of {len(DOCS_TO_INDEX)}; rejected {r['rejected']}")
    print("  A trace expires. This one does not -- deleting a chunk means re-indexing.")
guard(_index)

## Section 4 &mdash; Retention is a decision

Not setting it is also a decision, and it is the one that gets made by default.

In [ ]:
RETENTION_DAYS = {"prompt": 0, "trace": 30, "vector_store": None}   # None = forever

def retention_review() -> list:
    """One row per store: how long it keeps data, and whether anybody chose that."""
    return [{"store": store, "days": days,
             "forever": days is None, "decided": days is not None}
            for store, days in RETENTION_DAYS.items()]


def undecided() -> list:
    return [r["store"] for r in retention_review() if not r["decided"]]

In [ ]:
# --- Self-check: Section 4
check("every store is reviewed",
      lambda: len(retention_review()) == 3)
check("one of them keeps data forever",
      lambda: any(r["forever"] for r in retention_review()))
check("and that is the one nobody decided",
      lambda: undecided() == ["vector_store"],
      "'forever' is what you get when the question is never asked")
check("the prompt keeps nothing, which is the only store safe by construction",
      lambda: RETENTION_DAYS["prompt"] == 0)
check("the trace has a number, so somebody chose it",
      lambda: RETENTION_DAYS["trace"] > 0)

guard(lambda: [print(f"  {r['store']:14} {str(r['days']):>6} days"
                     f"   {'CHOSEN' if r['decided'] else 'NOBODY DECIDED'}")
               for r in retention_review()])

## Run it for real &mdash; was the customer data ever load-bearing?

Send a redacted and an unredacted record to the model and ask each for one action.

In [ ]:
if llm_ready():
    def _does_pii_help():
        for label, payload in (("redacted  ", redact(RAW_LEDGER["PMT-1003"])),
                               ("full record", RAW_LEDGER["PMT-1003"])):
            reply = ask("You are a payments operations agent. Recommend one action for this "
                        "case in a single short sentence.\n\n"
                        + json.dumps(payload, default=str))
            print(f"  [{label}] {reply.strip()[:160]}")
    guard(_does_pii_help)

### Read it

If the two recommendations are the same &mdash; and they should be, because the decision turns on
`status` and `reason_code` &mdash; then every customer field you sent was pure liability. It bought
nothing, and it is now in the prompt, the trace, and anywhere else that context was copied.

That is the usual finding. The fields go in because the tool returned them and nobody filtered,
not because anything needed them.

**What you take from this lab:** decide the boundary once, as an allow-list; put it where the data
enters, which is the tool; then point the same decision at the trace handler and the index. And
give every store a retention number that a person chose.

In [ ]:
score()

## Your turn

1. `leaks` matches exact values, which is the easy case. Real leakage is paraphrase &mdash;
   &ldquo;the Sharma payment&rdquo;. What would you actually have to check, and can you check it cheaply?
2. Your trace needs to be debuggable. Replace redaction with a stable pseudonym per beneficiary,
   so a support engineer can follow one customer across runs without seeing a name. What have you
   just created, and where does the mapping live?
3. Attach `RedactingTracer` to the whole agent rather than one tool, and find out what else it
   sees. `on_llm_start` gets the rendered prompt; is your boundary in front of that too?